# 02 — Column-by-Column Exploration

Before any cleaning, each column is examined on its own — its shape, fill rate, distribution, and a few real values — so cleaning decisions are made with the data in view rather than guessed.

Approach: convert the raw JSONL once into a flattened Parquet store (fast to reload), then walk one column at a time. For each column: look at stats and a chart, note anything that needs cleaning, decide what to do. Cleaning code is added per column as decisions are made, not up front.

The 11 source columns: `uid`, `title`, `journal`, `pubdate`, `pubdate_raw`, `pubdate_precision`, `abstract_sections`, `authors`, `mesh_terms`, `keywords`, `coi_statement`.

## 1. Setup
Resolves paths (works from `notebooks/` or the project root) and lists the raw monthly files. Uses `orjson` for faster JSON parsing.

In [ ]:
import os, re, glob
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

try:
    import orjson
    def jloads(b):
        """Parse a JSON line (orjson if available, ~2-4x faster than stdlib)."""
        return orjson.loads(b)
except ImportError:
    import json
    def jloads(b):
        return json.loads(b)


def find_project_root() -> str:
    """Locate the project root (the directory containing ``data/``).

    Walks up from the working directory so the notebook runs from ``notebooks/`` or root.

    Returns:
        str: Absolute path to the project root.
    """
    cwd = os.getcwd()
    if os.path.basename(cwd) == "notebooks":
        return os.path.dirname(cwd)
    if os.path.isdir(os.path.join(cwd, "data")) or os.path.isdir(os.path.join(cwd, "notebooks")):
        return cwd
    p = cwd
    for _ in range(5):
        if os.path.isdir(os.path.join(p, "data")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return cwd


ROOT = find_project_root()
RAW_DATA_DIR = os.path.join(ROOT, "data", "0_raw", "results")   # input: monthly JSONL
FLAT_DIR     = os.path.join(ROOT, "data", "1_flat")             # fast store: flattened Parquet
os.makedirs(FLAT_DIR, exist_ok=True)
files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "results_*.jsonl")))
assert files, f"no results_*.jsonl in {RAW_DATA_DIR}"
print(f"root: {ROOT}\ninput files: {len(files)}")

## 2. Build the Parquet store (run once)
Converts the raw JSONL into a flattened, columnar Parquet store one time, streaming in batches so memory stays flat. Nested fields (abstract, authors, MeSH, keywords) are flattened and helper columns derived. No rows are dropped and no text is altered beyond joining the abstract sections — it is a faithful, fast-loading mirror of the raw data. Re-running rebuilds it; skip this cell on later sessions once the store exists.

In [ ]:
FLAT_SCHEMA = pa.schema([
    ("uid", pa.string()), ("title", pa.string()), ("journal", pa.string()),
    ("year", pa.int16()), ("pubdate", pa.string()), ("pubdate_precision", pa.string()),
    ("abstract", pa.string()), ("abstract_len", pa.int32()),
    ("author_names", pa.list_(pa.string())), ("affiliations", pa.list_(pa.string())), ("n_authors", pa.int32()),
    ("mesh_descriptors", pa.list_(pa.string())), ("n_mesh", pa.int32()),
    ("keywords", pa.list_(pa.string())), ("n_keywords", pa.int32()),
    ("coi_statement", pa.string()), ("has_coi", pa.bool_()), ("source_month", pa.string()),
])


def flatten_abstract(sections) -> str:
    """Join structured abstract sections into one string (labels prefixed).

    Args:
        sections (list): The raw ``abstract_sections`` value.

    Returns:
        str: The abstract text, or ``""`` if empty/invalid.
    """
    if not isinstance(sections, list):
        return ""
    out = []
    for sec in sections:
        if isinstance(sec, dict) and sec.get("text"):
            label = (sec.get("label") or "").strip()
            out.append(f"{label}: {sec['text']}" if label else sec["text"])
    return " ".join(out)


def build_flat_store(batch_size: int = 100_000) -> int:
    """Convert the raw JSONL once into a flattened Parquet store for fast exploration.

    Streams every record, flattens the nested fields (abstract, authors, MeSH, keywords)
    and derives helper columns, writing Parquet shards in batches so memory stays flat.
    No rows are dropped and no text is altered beyond joining the abstract sections, so
    the store is a faithful, fast-loading mirror of the raw data.

    Args:
        batch_size (int): Rows accumulated before each Parquet shard is written.

    Returns:
        int: Number of shards written to ``FLAT_DIR``.
    """
    buf, idx = [], 0

    def flush(buffer, index):
        if not buffer:
            return index
        pq.write_table(pa.Table.from_pylist(buffer, schema=FLAT_SCHEMA),
                       os.path.join(FLAT_DIR, f"flat_{index:05d}.parquet"), compression="zstd")
        return index + 1

    for fp in tqdm(files, desc="JSONL -> Parquet"):
        m = re.search(r"results_(\d{4})_(\d{2})", fp)
        month = f"{m.group(1)}-{m.group(2)}" if m else ""
        with open(fp, "rb") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                r = jloads(line)
                authors = r.get("authors") or []
                names = [a.get("name", "") for a in authors if isinstance(a, dict) and a.get("name")]
                affs = [x for a in authors if isinstance(a, dict) for x in (a.get("affiliations") or [])]
                mesh = [t.get("descriptor", "") for t in (r.get("mesh_terms") or [])
                        if isinstance(t, dict) and t.get("descriptor")]
                kws = [k for k in (r.get("keywords") or []) if k]
                coi = (r.get("coi_statement") or "").strip()
                abstract = flatten_abstract(r.get("abstract_sections"))
                pd_str = r.get("pubdate", "")
                buf.append({
                    "uid": str(r.get("uid", "")), "title": r.get("title", ""),
                    "journal": r.get("journal", ""),
                    "year": int(pd_str[:4]) if pd_str[:4].isdigit() else None,
                    "pubdate": pd_str, "pubdate_precision": r.get("pubdate_precision", ""),
                    "abstract": abstract, "abstract_len": len(abstract),
                    "author_names": names, "affiliations": affs, "n_authors": len(names),
                    "mesh_descriptors": mesh, "n_mesh": len(mesh),
                    "keywords": kws, "n_keywords": len(kws),
                    "coi_statement": coi, "has_coi": bool(coi), "source_month": month,
                })
                if len(buf) >= batch_size:
                    idx = flush(buf, idx); buf = []
    idx = flush(buf, idx)
    return idx


n_shards = build_flat_store()
print(f"wrote {n_shards} Parquet shard(s) to {FLAT_DIR}")

 stage 1: the same data flattened into Parquet (nested fields unpacked, fast to load, nothing dropped)

## 3. Load
Loads the Parquet store into a DataFrame. The whole corpus loads in seconds; for a single statistic, read only the columns needed (near-instant, minimal memory) — see the commented example.

In [ ]:
# Whole corpus (seconds to load; a few GB in memory):
df = pd.read_parquet(FLAT_DIR)

# For a single statistic, read only the columns needed (near-instant, tiny memory), e.g.:
#   pd.read_parquet(FLAT_DIR, columns=["year", "abstract_len"])

print(f"loaded {len(df):,} rows  |  columns: {list(df.columns)}")
df.head(3)

## 4. Data Inspection
